# SRQ-FLY Priority 2A — allocation-bounded blocked QR
Synthetic Tesla-T4 benchmark only. No dataset, feature cache, or held-out test is opened.

In [ ]:
# === Edit path/source values only ===
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
PERSIST_TO_DRIVE=False  # True uses the small resumable Drive output below.
DRIVE_OUTPUT='/content/drive/MyDrive/T-SOHO/srq_priority2a_output'
LOCAL_OUTPUT='/content/srq_priority2a_output'

In [ ]:
# Clone/update the locked branch from a valid parent directory.
import os, subprocess, sys
from pathlib import Path
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir('/content')
repo=Path(WORK_DIR)
if (repo/'.git').is_dir():
    os.chdir(repo)
    subprocess.run(['git','fetch','origin',REPO_BRANCH],check=True)
    subprocess.run(['git','checkout',REPO_BRANCH],check=True)
    subprocess.run(['git','pull','--ff-only','origin',REPO_BRANCH],check=True)
else:
    assert not repo.exists(),f'Non-git path already exists: {repo}'
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
    os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest'],check=True)
OUTPUT_DIR=DRIVE_OUTPUT if PERSIST_TO_DRIVE else LOCAL_OUTPUT
print('REPO:',Path.cwd())
print('OUTPUT:',OUTPUT_DIR)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
# Immutable source/config identities.
import hashlib, json
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/srq_fly_priority2a_memory.json'
RUNNER='tools/srq_fly_priority2_memory_benchmark.py'
EXPECTED={
  CONFIG:'eb7da791122a385da39731fbaea82204eb295771f2abe46b4f92922703c6070c',
  RUNNER:'f361b26c0da5455c874590d66cb980e8b33e6898b28831e5060624f460265b93',
  'tools/srq_fly_system_benchmark.py':'f93b10b4b76f98d07bb7b0e285abbaa7e7babda06117427392bd25183b977d4f',
  'methods/srq_fly_optimized/learner.py':'59d2f81e0e0a9074468ecdcfd49e49db0c7d83fe7a67c0b74dc03071f58fc88b',
  'methods/srq_fly_optimized/storage.py':'99c4dab6f5c9e3da249f8c4dbf5cb9e5e303d8fd59f11e9d184db0d180bdf330',
}
for path,expected in EXPECTED.items():
    observed=sha(path); print(path,observed); assert observed==expected,(path,observed,expected)
dirty=subprocess.check_output(['git','status','--porcelain'],text=True).strip()
assert not dirty,f'Repository must be clean before benchmark:\n{dirty}'
assert json.loads(Path(CONFIG).read_text())['seed']==2025
print('PRIORITY-2A IDENTITY GATE: PASS')

In [ ]:
# CPU correctness, checkpoint, input-ownership, and protocol tests.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_fly_optimized.py','tests/test_srq_fly_priority2_memory.py']
completed=subprocess.run(command)
assert completed.returncode==0,'Correctness gate failed; return the full traceback.'
print('PRIORITY-2A CORRECTNESS GATE: PASS')

In [ ]:
# Repeated isolated T4 benchmark. Safe to rerun: completed workers resume.
assert __import__('torch').cuda.is_available(),'Select a GPU runtime first.'
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u',RUNNER,'--config',CONFIG,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('PRIORITY-2A START: 1 warm-up + 7 measured rounds; 6 workers per round.',flush=True)
print('Each worker prints START, two TASK lines, and DONE. Existing valid workers print RESUME.',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'Priority-2A runner failed; return the complete traceback.'
RESULT=Path(OUTPUT_DIR)/'priority2a_memory_results.json'
payload=json.loads(RESULT.read_text())
print('PRIORITY-2A STATUS:',payload['status'])
print('SELECTED:',payload['selected_candidate'])

In [ ]:
# Compact timing/memory decision table.
import pandas as pd
rows=[]
for item in payload['summaries']:
    rows.append({
      'method':item['label'],
      'median_update_s':item['update_seconds']['median'],
      'median_peak_GiB':item['peak_allocated_bytes']['median']/2**30,
      'persistent_MiB':item['persistent_state_bytes']/2**20,
      'time/exact':item.get('paired_update_ratio_to_exact',{}).get('median'),
      'peak/exact':item.get('paired_peak_allocated_ratio_to_exact',{}).get('median'),
      'max_logit_drift':item.get('maximum_relative_logit_drift_from_unchunked'),
      'all_gates':all(item.get('gates',{}).values()) if item.get('gates') else None,
    })
display(pd.DataFrame(rows))
if payload['status']=='STOP_MEMORY_GATE':
    print('STOP: chunking alone did not meet the locked peak-memory target. Do not relax gates.')
else:
    print('PASS: return the ZIP before any dataset experiment.')

In [ ]:
# Show per-stage CUDA peaks from the first measured run.
stage_rows=[]
for item in payload['summaries']:
    profiles=item.get('profiled_task_stage_cuda_memory')
    if not profiles: continue
    for task_index,task in enumerate(profiles,1):
        for stage,memory in task.items():
            stage_rows.append({'method':item['label'],'task':task_index,'stage':stage,'peak_GiB':memory['peak_allocated_bytes']/2**30})
stage_table=pd.DataFrame(stage_rows)
display(stage_table.sort_values(['method','task','peak_GiB'],ascending=[True,True,False]))

In [ ]:
# Export immutable evidence (small JSON/probe bundle).
import shutil
bundle=Path('/content/srq_fly_priority2a_memory')
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(OUTPUT_DIR,bundle/'output')
shutil.copy2(CONFIG,bundle/'locked_config.json')
shutil.copy2('docs/research/SRQ_FLY_PRIORITY2A_PROTOCOL.md',bundle/'protocol.md')
archive=shutil.make_archive('/content/srq_fly_priority2a_memory','zip',bundle.parent,bundle.name)
print('ZIP:',archive,'bytes=',Path(archive).stat().st_size,'sha256=',sha(archive))
from google.colab import files
files.download(archive)